---
# Commande pour le rendu : quarto render main.ipynb --execute
# Instructions pour quarto
title: "Traitement des données DVF Île-de-France"
format:
  html:
    code-fold: true
    embed-resources: true
---

# Introduction

- **Contexte :** nous exploitons les données des **Demandes de Valeurs Foncières (DVF)** pour analyser les prix des logements (appartements et maisons) en Île-de-France. Ces données fournissent des informations issues des actes notariés et des données cadastrales : valeur du bien, date de mutation, type de bien, surface, localisation, etc.
- **Objectif :** obtenir des **prix immobiliers précis au mètre carré**, par adresse et type de bien, afin de faciliter l’analyse du marché immobilier et son lien éventuel avec l’offre de transport.
- **Données :** nous utilisons les fichiers DVF couvrant les **cinq dernières années**. Ils sont disponibles en **format texte standardisé**, et la documentation complète est consultable [ici](https://eu.ftp.opendatasoft.com/stif/GTFS/opendata_gtfs.pdf).
    - Nettoyage des colonnes essentielles (code département, code postal, adresse)
    - Filtrage pour les départements d’Île-de-France
    - Reconstruction des adresses
    - Agrégation des transactions par adresse et date pour gérer les lots multiples
    - Calcul du **prix au mètre carré**
    - Filtrage des valeurs extrêmes pour éliminer les anomalies
    - Géolocalisation des logements via l’API Base Adresse Nationale (BAN)

# Chargement des données


In [ ]:
import sys
from pathlib import Path
import download_data
import pandas as pd
import numpy as np

script_path = Path.cwd().parent / "script"
sys.path.append(str(script_path))

file_names = download_data.get_valeur_fonciere_path()

with open(file_names, "r", encoding="utf-8") as f:
    for i in range(10):
        print(f.readline().strip())

df = pd.read_csv(file_names, sep="|")

colonnes_utiles = [
    "Date mutation", 
    "Nature mutation", 
    "Valeur fonciere", 
    "No voie", 
    "Voie",
    "Type de voie",
    "Code voie",
    "Code postal", 
    "Commune",
    "B/T/Q",
    "Code commune",
    "Code departement",
    "Type local", 
    "Code type local",
    "Surface reelle bati", 
    "Nombre pieces principales", 
    "Surface terrain", 
    "1er lot",
    "2eme lot",
    "3eme lot",
    "4eme lot",
    "5eme lot"
]

df_filtre = df[colonnes_utiles]

Utilisation des données en cache dans cache/valeur_fonciere
Identifiant de document|Reference document|1 Articles CGI|2 Articles CGI|3 Articles CGI|4 Articles CGI|5 Articles CGI|No disposition|Date mutation|Nature mutation|Valeur fonciere|No voie|B/T/Q|Type de voie|Code voie|Voie|Code postal|Commune|Code departement|Code commune|Prefixe de section|Section|No plan|No Volume|1er lot|Surface Carrez du 1er lot|2eme lot|Surface Carrez du 2eme lot|3eme lot|Surface Carrez du 3eme lot|4eme lot|Surface Carrez du 4eme lot|5eme lot|Surface Carrez du 5eme lot|Nombre de lots|Code type local|Type local|Identifiant local|Surface reelle bati|Nombre pieces principales|Nature culture|Nature culture speciale|Surface terrain
|||||||000001|02/01/2024|Vente|346,50||||B020|LE DELIVRE|1230|CHALEY|01|76||B|514||||||||||||0||||||P||99
|||||||000002|03/01/2024|Vente|10000,00||||B007|CHEVRY DESSOUS|1170|CHEVRY|01|103||B|1782||||||||||||0||||||S||115
|||||||000001|08/01/2024|Vente|249000,00||||B086|PIN HAMEAU|1290

/tmp/ipykernel_119049/640163238.py:16: DtypeWarning: Columns (18,23,24,26,28,30,31,33,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_names, sep="|")


# Traitement 

# Filtrage pour les départements d’Île-de-France

Les données DVF sont nationale et nous souhaitons concentrer nos analyse sur l'Île de France. Nous filtrons les département après avoir vérifié l'absence de valeurs manquantes dans la colonne du dataset et nettoyé celle-ci. 

In [4]:

#Nettoyage de la Colonne Code departement
df_filtre["Code departement"] = df_filtre["Code departement"].astype(str).str.strip()
# Filtrage des communes en ile de france 
codes_idf = ["75", "77", "78", "91", "92", "93", "94", "95"]
idf = df_filtre[df_filtre["Code departement"].isin(codes_idf)]

/tmp/ipykernel_119049/1379743959.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtre["Code departement"] = df_filtre["Code departement"].astype(str).str.strip()


# Reconstruction des adresses 

Nous reconstruson l'adresse des logement à partir des information mis à notre disposition : Numéro de voie, voie et type de voie

In [5]:
idf["x"] = pd.NA
idf["y"] = pd.NA

idf["Code postal"] = idf["Code postal"].astype(str).str.replace(".0", "", regex=False).str.strip()
idf["No voie"] = idf["No voie"].astype(str).str.replace(".0", "", regex=False).str.strip()

idf["Type de voie"] = idf["Type de voie"].fillna("").astype(str).str.strip()
idf["Voie"] = idf["Voie"].fillna("").astype(str).str.strip()

# Créer une colonne adresse propre
idf["adresse"] = (
    idf["No voie"] + " " +
    idf["Type de voie"] + " " +
    idf["Voie"]
).str.replace(" +", " ", regex=True).str.strip()

/tmp/ipykernel_119049/706913884.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  idf["x"] = pd.NA
/tmp/ipykernel_119049/706913884.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  idf["y"] = pd.NA
/tmp/ipykernel_119049/706913884.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-vie

# Agrégation des transactions par adresse et date pour gérer les lots multiples

Certaines transactions immobilières dans les fichiers DVF concernent plusieurs lots (par exemple, un immeuble vendu en plusieurs parties ou un bien cadastral divisé).  
Si chaque lot était conservé séparément, cela entraînerait un **double comptage** des biens et biaiserait les calculs de surface totale, de nombre de pièces et du prix au mètre carré.  

Pour corriger cela, nous effectuons une **agrégation par adresse et date de mutation**.  
Chaque groupe de lignes correspondant à une même adresse et à la même date est considéré comme une **transaction unique**.  

Lors de cette agrégation :  
- La **valeur foncière** est soit conservée telle quelle si identique pour tous les lots, soit sommée pour refléter l’ensemble des lots.  
- Le **code postal** et la **commune** sont pris à partir de la première valeur du groupe, car elles sont identiques pour tous les lots.  
- La **surface réelle bâtie**, la **surface du terrain** et le **nombre de pièces principales** sont **sommées** pour obtenir les totaux correspondant à l’ensemble des lots.  
- Le **code commune** et le **type de local** sont pris à partir de la première valeur non manquante, afin de préserver l’information principale du bien.  

Cette opération garantit que chaque transaction est comptabilisée **une seule fois**, et que les mesures de surface et de composition du bien sont exactes.  
C’est une étape essentielle pour calculer un **prix au mètre carré fiable** et pour réaliser des analyses statistiques précises.

In [ ]:
def valeur_fonciere_agg(x):
    if len(x.unique()) == 1:
        return x.iloc[0]
    else:
        return pd.to_numeric(x, errors='coerce').sum()

def sum_numeric(x):
    return pd.to_numeric(x, errors='coerce').sum()

idf = (
    idf
    .groupby(["adresse", "Date mutation"], as_index=False)
    .agg({
        "Valeur fonciere": valeur_fonciere_agg,
        "Code postal": "first",
        "Commune": "first",
        "Surface reelle bati": sum_numeric,
        "Surface terrain": sum_numeric,
        "Nombre pieces principales": sum_numeric,
        "Code commune": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else pd.NA,
        "Type local": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else pd.NA
    })
)

# Nettoyage des données et traitement des valeurs aberrantes

Cette étape vise à nettoyer les données DVF et à éliminer les **valeurs aberrantes** afin d’obtenir un indicateur de **valeur foncière au mètre carré** fiable et exploitable pour l’analyse.

## Filtrage des biens et des surfaces
- Suppression des observations dont la **surface réelle bâtie** est manquante.
- Conversion de la surface réelle bâtie en format numérique.
- Conservation uniquement des biens dont la surface est comprise entre **9 m² et 300 m²**, afin d’exclure les valeurs irréalistes.
- Sélection des **maisons et appartements** uniquement, pour se concentrer sur le résidentiel.
- Suppression des lignes avec une **valeur foncière manquante**.

## Nettoyage de la valeur foncière
La variable **valeur foncière** est nettoyée pour garantir un format numérique homogène 

## Calcul de la valeur foncière au mètre carré
La **valeur foncière au mètre carré** est calculée comme le rapport entre la valeur foncière et la surface réelle bâtie.

## Analyse des distributions avant filtrage
- Calcul des **déciles** de la valeur foncière au mètre carré afin d’observer la distribution.
- Identification des valeurs minimale et maximale.
- Comptage du nombre d’observations avant filtrage.

## Traitement des valeurs aberrantes
Afin d’éliminer les prix au mètre carré incohérents ou extrêmes :
- Conservation uniquement des biens dont la valeur foncière au mètre carré est comprise entre **1 000 € et 20 000 €**.
- Suppression des observations situées en dehors de cet intervalle.

## Analyse après filtrage
- Recalcul des **déciles** après suppression des valeurs aberrantes.
- Comparaison du nombre d’observations avant et après filtrage.
- Vérification des nouvelles valeurs minimale et maximale.

Cette démarche permet de réduire l’influence des valeurs extrêmes, d’améliorer la robustesse des statistiques descriptives et d’assurer la fiabilité des analyses de prix immobiliers.


In [ ]:
 idf = idf[idf["Surface reelle bati"].notna()]
 idf["Surface reelle bati"] = pd.to_numeric(idf["Surface reelle bati"])
 idf = idf[(idf["Surface reelle bati"] >= 9) & (idf["Surface reelle bati"] <= 300)]
 
 idf = idf[idf["Type local"].isin(["Maison", "Appartement"])]
 
 idf = idf.dropna(subset=["Valeur fonciere"])

idf["Valeur fonciere"] = (
    idf["Valeur fonciere"]
        .astype(str)
        .str.replace(" ", "", regex=False)    
        .str.replace(",", ".", regex=False)    
        .str.replace("\xa0", "", regex=False)  
        .str.extract(r'(\d+\.?\d*)')[0]        
        .astype(float)
)
idf["Valeur foncière au mètre carré"] = (
    idf["Valeur fonciere"] / idf["Surface reelle bati"]
)

deciles_avant = idf["Valeur foncière au mètre carré"].quantile([0.1 * i for i in range(1, 11)])
deciles_avant = deciles_avant.apply(lambda x: f"{x:,.2f} €")

n_before = len(idf)
prix_min_avant = idf["Valeur foncière au mètre carré"].min()
prix_max_avant = idf["Valeur foncière au mètre carré"].max()


idf = idf.loc[
    (idf["Valeur foncière au mètre carré"] >= 1000) &
    (idf["Valeur foncière au mètre carré"] <= 20000)
]

deciles_apres = idf["Valeur foncière au mètre carré"].quantile([0.1 * i for i in range(1, 11)])
deciles_apres = deciles_apres.apply(lambda x: f"{x:,.2f} €")

n_after = len(idf)
n_supprimees = n_before - n_after
prix_min_apres = idf["Valeur foncière au mètre carré"].min()
prix_max_apres = idf["Valeur foncière au mètre carré"].max()

In [ ]:
print(f"Nombre de lignes avant filtrage : {n_before}")
print(f"Valeur foncière au mètre carré minimale avant filtrage : {prix_min_avant}")
print(f"Valeur foncière au mètre carré maximal avant filtrage : {prix_max_avant}")

In [ ]:
print("Déciles avant filtrage :")
print(deciles_avant.to_string())

In [ ]:
print(f"Nombre de lignes supprimées : {n_supprimees}")
print(f"Valeur foncière au mètre carré minimale après filtrage : {prix_min_apres}")
print(f"Valeur foncière au mètre carré maximal avant filtrage : {prix_max_apres}")

In [ ]:
print("\nDéciles après filtrage :")
print(deciles_apres.to_string())